# Autoencoder Analysis — SailGP Telemetry

Five autoencoder experiments on SailGP boat telemetry with shared evaluation:
reconstruction stats, latent visualization, downstream metrics, PCA baselines, and null-checks.

**Data:** Bermuda (2026) + Halifax (2024) · 1 Hz F50 telemetry

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "dataExploration" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataExploration.autoencoder_experiments import (
    load_prepared_data,
    run_all_experiments,
    run_experiment_1,
    run_experiment_2,
    run_experiment_3,
    run_experiment_4,
    run_experiment_5,
    null_check_auc,
)

sns.set_theme(style="whitegrid")
FAST = False  # set True for smoke test (fewer epochs)
VERBOSE = True  # tqdm epoch bars + batch checkpoints during training

In [ ]:
df = load_prepared_data()
print(f"Total rows: {len(df):,}")
print(f"Racing rows: {(df['TRK_BOAT_RACE_STATUS_unk']==2).sum():,}")
print(f"Venues: {df['venue'].unique().tolist()}")
df.head(2)

## Experiment 1 — Foiling-Mode AE

Deep AE on control surfaces + ride heights + platform. **Meaningful if** foiling rows reconstruct with lower error (AUC > 0.6 vs null check).

In [ ]:
exp1 = run_experiment_1(df, epochs=30 if FAST else 100, verbose=VERBOSE)
print("Metrics:", exp1["metrics"])
print("Meaningful:", exp1["meaningful"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
test_df = exp1["test_df"]
for ax, col, title in [
    (axes[0], "foiling", "Recon error by foiling"),
    (axes[1], "team", "Recon error by team (top 8)"),
]:
    if col == "team":
        top = test_df.groupby("team")["recon_error"].mean().nlargest(8).index
        sub = test_df[test_df["team"].isin(top)]
        sns.boxplot(data=sub, x="team", y="recon_error", ax=ax)
    else:
        sns.histplot(data=test_df, x="recon_error", hue=col, ax=ax, kde=True)
    ax.set_title(title)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(exp1["losses"])
plt.title("Exp1 training loss")
plt.xlabel("epoch")
plt.show()

## Experiment 2 — Sailing-Mode Discovery (LSTM AE)

30-second windows of wind + control surfaces. **Meaningful if** LSTM-AE silhouette beats PCA and clusters align to legs/TWA bins.

In [ ]:
exp2 = run_experiment_2(df, epochs=50 if FAST else 200, verbose=VERBOSE)
print("Metrics:", exp2["metrics"])
print("Meaningful:", exp2["meaningful"])

meta = exp2["meta_test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "leg" in meta.columns:
    pd.crosstab(meta["cluster"], meta["leg"], normalize="index").plot(kind="bar", ax=axes[0], stacked=True)
    axes[0].set_title("Cluster vs leg (row-normalized)")
if "twa_bin" in meta.columns:
    pd.crosstab(meta["cluster"], meta["twa_bin"], normalize="index").plot(kind="bar", ax=axes[1], stacked=True)
    axes[1].set_title("Cluster vs TWA bin")
plt.tight_layout()
plt.show()

try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=42)
    emb = reducer.fit_transform(exp2["latent_test"])
    plot_df = meta.copy()
    plot_df["u1"], plot_df["u2"] = emb[:, 0], emb[:, 1]
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=plot_df, x="u1", y="u2", hue="twa_bin", palette="tab10", s=15, alpha=0.7)
    plt.title("LSTM-AE latent (UMAP) colored by TWA bin")
    plt.show()
except ImportError:
    print("umap-learn not installed; skip UMAP plot")

## Experiment 3 — Team DNA (VAE)

Shared VAE across teams; compare latent centroids and rank correlation. **Meaningful if** distance to top team correlates with mean rank (Spearman ρ < -0.3).

In [ ]:
exp3 = run_experiment_3(df, epochs=40 if FAST else 150, verbose=VERBOSE)
print("Metrics:", exp3["metrics"])
print("Top team:", exp3["top_team"])
print("Meaningful:", exp3["meaningful"])

dist_df = pd.DataFrame(exp3["distance_matrix"], index=exp3["teams"], columns=exp3["teams"])
plt.figure(figsize=(8, 6))
sns.heatmap(dist_df, cmap="viridis", annot=False)
plt.title("Team latent centroid distances")
plt.show()

var_df = pd.DataFrame({"team": list(exp3["variances"].keys()), "latent_var": list(exp3["variances"].values())})
var_df = var_df.sort_values("latent_var")
plt.figure(figsize=(8, 4))
sns.barplot(data=var_df, x="team", y="latent_var", color="steelblue")
plt.xticks(rotation=45)
plt.title("Per-team latent variance (consistency)")
plt.tight_layout()
plt.show()

## Experiment 4 — Tactical Anomaly Detection

Train AE on clean racing; score pre-start and penalty rows. **Meaningful if** penalty rows have higher error (Mann-Whitney p < 0.05).

In [ ]:
exp4 = run_experiment_4(df, epochs=30 if FAST else 80, verbose=VERBOSE)
print("Metrics:", exp4["metrics"])
print("Meaningful:", exp4["meaningful"])

racing = exp4["racing_scored"]
penalty_mask = racing["TRK_PENALTY_COUNT_unk"].fillna(0) > 0
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(racing.loc[~penalty_mask, "recon_error"], label="clean", kde=True, ax=ax)
sns.histplot(racing.loc[penalty_mask, "recon_error"], label="penalty", kde=True, ax=ax)
ax.legend()
ax.set_title("Reconstruction error: clean vs penalty racing rows")
plt.show()

pre = exp4["prestart_scored"]
if len(pre) and "PC_TTS_s" in pre.columns:
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=pre, x="PC_TTS_s", y="recon_error", estimator="mean", errorbar="sd")
    plt.gca().invert_xaxis()
    plt.title("Pre-start recon error vs time-to-start")
    plt.show()

display(exp4["top5_moments"])

## Experiment 5 — Cross-Venue Sparse AE

Train Bermuda→Halifax, Halifax→Bermuda, and joint. **Meaningful if** latent space mixes venues by wind regime (mixing score > 0.25).

In [ ]:
exp5 = run_experiment_5(df, epochs=40 if FAST else 100, verbose=VERBOSE)
print("Metrics:", exp5["metrics"])
print("Meaningful:", exp5["meaningful"])

joint = exp5["conditions"]["joint"]["metrics"]
z_berm = joint["Bermuda"]["latent"]
z_hal = joint["Halifax"]["latent"]
labels = np.array(["Bermuda"] * len(z_berm) + ["Halifax"] * len(z_hal))
z_all = np.vstack([z_berm, z_hal])

try:
    import umap
    emb = umap.UMAP(n_components=2, random_state=42).fit_transform(z_all)
    umap_df = pd.DataFrame({"u1": emb[:, 0], "u2": emb[:, 1], "venue": labels})
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=umap_df, x="u1", y="u2", hue="venue", alpha=0.3, s=8)
    plt.title("Joint sparse-AE latent: Bermuda vs Halifax")
    plt.show()
except ImportError:
    print("umap-learn not installed")

feat_delta = joint["feature_error_delta"]
feat_df = pd.DataFrame({"feature": list(feat_delta.keys()), "delta": list(feat_delta.values())})
feat_df = feat_df.reindex(feat_df["delta"].abs().sort_values(ascending=False).index)
plt.figure(figsize=(10, 4))
sns.barplot(data=feat_df.head(12), x="feature", y="delta", color="coral")
plt.xticks(rotation=60, ha="right")
plt.title("Feature recon error delta (Halifax - Bermuda)")
plt.tight_layout()
plt.show()

## Summary & Null Checks

Compare all experiments. Null checks shuffle labels to verify findings are not random.

In [ ]:
summary_rows = [
    {
        "experiment": exp1["experiment"],
        "meaningful": exp1["meaningful"],
        "primary_metric": f"auc_test={exp1['metrics']['auc_test']:.3f}",
        "null_check": f"null_auc={exp1['metrics']['null_auc']:.3f}",
        "passes_null": exp1["metrics"]["auc_test"] > exp1["metrics"]["null_auc"] + 0.05,
    },
    {
        "experiment": exp2["experiment"],
        "meaningful": exp2["meaningful"],
        "primary_metric": f"sil={exp2['metrics']['ae_silhouette']:.3f}",
        "null_check": f"pca_sil={exp2['metrics']['pca_silhouette']:.3f}",
        "passes_null": exp2["metrics"]["ae_silhouette"] > exp2["metrics"]["pca_silhouette"],
    },
    {
        "experiment": exp3["experiment"],
        "meaningful": exp3["meaningful"],
        "primary_metric": f"rho={exp3['metrics']['rank_spearman_rho']:.3f}",
        "null_check": f"p={exp3['metrics']['rank_spearman_p']:.4f}",
        "passes_null": exp3["metrics"]["rank_spearman_p"] < 0.1,
    },
    {
        "experiment": exp4["experiment"],
        "meaningful": exp4["meaningful"],
        "primary_metric": f"mwu_p={exp4['metrics']['mannwhitney_p']:.4f}",
        "null_check": "penalty>clean" if exp4["metrics"]["penalty_mean_error"] > exp4["metrics"]["clean_mean_error"] else "no signal",
        "passes_null": exp4["metrics"]["mannwhitney_p"] < 0.05,
    },
    {
        "experiment": exp5["experiment"],
        "meaningful": exp5["meaningful"],
        "primary_metric": f"mixing={exp5['metrics']['venue_mixing_score']:.3f}",
        "null_check": f"B→H ratio={exp5['metrics']['bermuda_to_halifax_ratio']:.2f}",
        "passes_null": exp5["metrics"]["venue_mixing_score"] > 0.25,
    },
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df.style.background_gradient(subset=["meaningful"], cmap="RdYlGn"))

meaningful_count = summary_df["meaningful"].sum()
print(f"\n{meaningful_count}/5 experiments show meaningful structure.")